# Clustering Review

Use this notebook after running `scripts/07_clustering.py` to inspect UMAP structure, cluster sizes, sample mixing, and QC metric overlays.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path("..").resolve()
CLUSTERING_DIR = REPO_ROOT / "results" / "clustering"
PLOTS_DIR = REPO_ROOT / "plots" / "clustering"

adata = ad.read_h5ad(CLUSTERING_DIR / "clustered.h5ad")
adata

In [ ]:
display(Image(filename=PLOTS_DIR / "umap_leiden.png"))
display(Image(filename=PLOTS_DIR / "umap_sample_id.png"))

In [ ]:
cluster_counts = adata.obs["leiden"].value_counts().sort_index()
cluster_counts.to_frame("n_cells")

In [ ]:
sample_counts = adata.obs["sample_id"].value_counts().sort_index()
sample_counts.to_frame("n_cells")

In [ ]:
cluster_by_sample = pd.crosstab(adata.obs["leiden"], adata.obs["sample_id"])
cluster_by_sample

In [ ]:
cluster_by_sample_fraction = cluster_by_sample.div(cluster_by_sample.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
cluster_by_sample_fraction.plot(kind="bar", stacked=True, ax=ax, width=0.9)
ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Fraction of cells")
ax.legend(title="sample_id", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()

In [ ]:
try:
    import scanpy as sc

    sc.pl.umap(
        adata,
        color=["leiden", "sample_id", "total_counts", "n_genes_by_counts", "pct_counts_mt"],
        frameon=False,
        wspace=0.4,
    )
except ImportError:
    print("Install scanpy to draw interactive UMAP overlays in this notebook.")

In [ ]:
qc_by_cluster = adata.obs.groupby("leiden")[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].median()
qc_by_cluster

Review notes:

- Clusters dominated by one sample may reflect biology, batch effects, or sample-specific quality issues.
- Clusters with unusually high `total_counts` or `n_genes_by_counts` may contain doublets.
- Clusters with high `pct_counts_mt` may contain stressed or dying cells.
- If the resolution is too coarse or too fine, rerun `scripts/07_clustering.py --resolution <value>`.